# 特定投資株式情報 クイック分析

このnotebookは、有価証券報告書から特定投資株式情報を素早く抽出・分析するための簡潔版です。

## 1. セットアップ

In [ ]:
# 必要なライブラリのインポート
import sys
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 既存のスクリプトをインポート
sys.path.append('/Users/satoki252595/work/0000_kabulab/0001_MarketableSecuritiesAnalysis')
from improved_extraction import ImprovedMarketableSecuritiesExtractor

print("セットアップ完了")

## 2. パラメータ設定

In [ ]:
# 設定
XBRL_FOLDER_PATH = "/Users/satoki252595/work/0000_kabulab/0001_MarketableSecuritiesAnalysis/xbrl"
LIMIT_FOLDERS = 20  # 処理するフォルダ数の制限（テスト用）
OUTPUT_FILE = f"quick_analysis_{LIMIT_FOLDERS}folders.csv"

# フォルダ数確認
folders = [f for f in os.listdir(XBRL_FOLDER_PATH) 
          if os.path.isdir(os.path.join(XBRL_FOLDER_PATH, f))
          and not f.startswith('.')]

print(f"利用可能フォルダ数: {len(folders)}")
print(f"処理予定フォルダ数: {min(LIMIT_FOLDERS, len(folders))}")
print(f"出力ファイル: {OUTPUT_FILE}")

## 3. データ抽出実行

In [ ]:
# 抽出器を初期化
extractor = ImprovedMarketableSecuritiesExtractor(XBRL_FOLDER_PATH)

# データ抽出実行
print("データ抽出を開始します...")
extractor.process_all_folders(limit=LIMIT_FOLDERS)

# 結果をDataFrameに変換
df = extractor.to_dataframe()

print(f"\n=== 抽出結果 ===")
print(f"抽出データ件数: {len(df)}")
if not df.empty:
    print(f"提出会社数: {df['filing_company_code'].nunique()}")
    print(f"保有銘柄数: {df['held_security_name'].nunique()}")

## 4. データ確認

In [ ]:
if not df.empty:
    print("=== データサンプル ===")
    display(df[['filing_company_code', 'held_security_name', 'held_shares', 'book_value_million_yen', 'holding_purpose']].head(10))
    
    print("\n=== 基本統計 ===")
    print(df['book_value_million_yen'].describe())
else:
    print("データが抽出されませんでした。")

## 5. 分析結果

In [ ]:
if not df.empty:
    print("=== 最も多く保有されている銘柄 Top 10 ===")
    top_securities = df['held_security_name'].value_counts().head(10)
    display(top_securities)
    
    print("\n=== 貸借対照表計上額 上位10銘柄 ===")
    top_values = df.nlargest(10, 'book_value_million_yen')[['held_security_name', 'book_value_million_yen', 'filing_company_code']]
    display(top_values)
    
    print("\n=== 保有目的の分析 ===")
    purpose_analysis = df['holding_purpose'].value_counts().head(5)
    for purpose, count in purpose_analysis.items():
        print(f"{count}件: {purpose[:100]}...")

## 6. 簡単なビジュアライゼーション

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 貸借対照表計上額の分布
    axes[0, 0].hist(df['book_value_million_yen'], bins=30, alpha=0.7, color='skyblue')
    axes[0, 0].set_title('Book Value Distribution')
    axes[0, 0].set_xlabel('Amount (Million Yen)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_yscale('log')
    
    # 2. 上位保有銘柄
    top_10 = df['held_security_name'].value_counts().head(10)
    axes[0, 1].barh(range(len(top_10)), top_10.values, color='lightgreen')
    axes[0, 1].set_yticks(range(len(top_10)))
    axes[0, 1].set_yticklabels([name[:20] + '...' if len(name) > 20 else name for name in top_10.index])
    axes[0, 1].set_title('Top 10 Most Held Securities')
    axes[0, 1].set_xlabel('Count')
    
    # 3. 提出会社別の保有銘柄数
    company_counts = df['filing_company_code'].value_counts().head(10)
    axes[1, 0].bar(range(len(company_counts)), company_counts.values, color='coral')
    axes[1, 0].set_title('Holdings by Company')
    axes[1, 0].set_xlabel('Company Rank')
    axes[1, 0].set_ylabel('Number of Holdings')
    
    # 4. 貸借対照表計上額上位
    top_values = df.nlargest(10, 'book_value_million_yen')
    axes[1, 1].barh(range(len(top_values)), top_values['book_value_million_yen'], color='gold')
    axes[1, 1].set_yticks(range(len(top_values)))
    axes[1, 1].set_yticklabels([name[:15] + '...' if len(name) > 15 else name for name in top_values['held_security_name']])
    axes[1, 1].set_title('Top 10 by Book Value')
    axes[1, 1].set_xlabel('Amount (Million Yen)')
    
    plt.tight_layout()
    plt.show()
else:
    print("グラフ表示用のデータがありません。")

## 7. 結果保存

In [ ]:
# CSVファイルに保存
if not df.empty:
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    print(f"データを保存しました: {OUTPUT_FILE}")
    print(f"ファイルサイズ: {os.path.getsize(OUTPUT_FILE) / 1024:.2f} KB")
    print(f"総レコード数: {len(df)}")
    
    # サマリーファイルも作成
    summary_file = f"summary_{LIMIT_FOLDERS}folders.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"=== 処理結果サマリー ===\n")
        f.write(f"処理フォルダ数: {LIMIT_FOLDERS}\n")
        f.write(f"抽出データ件数: {len(df)}\n")
        f.write(f"提出会社数: {df['filing_company_code'].nunique()}\n")
        f.write(f"保有銘柄数: {df['held_security_name'].nunique()}\n")
        f.write(f"平均貸借対照表計上額: {df['book_value_million_yen'].mean():.2f} 百万円\n")
        f.write(f"最大貸借対照表計上額: {df['book_value_million_yen'].max():.2f} 百万円\n")
        f.write(f"\n=== 最も多く保有されている銘柄 Top 5 ===\n")
        for i, (name, count) in enumerate(df['held_security_name'].value_counts().head(5).items()):
            f.write(f"{i+1}. {name}: {count}件\n")
    
    print(f"サマリーファイルを作成しました: {summary_file}")
else:
    print("保存するデータがありません。")

print("\n処理完了！")

## 8. 再実行用セル（オプション）

In [ ]:
# より多くのフォルダを処理したい場合は、この数値を変更してください
NEW_LIMIT = 50
RUN_EXTENDED = False  # Trueにすると拡張処理を実行

if RUN_EXTENDED:
    print(f"拡張処理を開始します（{NEW_LIMIT}フォルダ）...")
    
    # 新しい抽出器で処理
    extended_extractor = ImprovedMarketableSecuritiesExtractor(XBRL_FOLDER_PATH)
    extended_extractor.process_all_folders(limit=NEW_LIMIT)
    
    # 結果保存
    extended_df = extended_extractor.to_dataframe()
    if not extended_df.empty:
        extended_filename = f"extended_analysis_{NEW_LIMIT}folders.csv"
        extended_df.to_csv(extended_filename, index=False, encoding='utf-8-sig')
        print(f"拡張分析結果を保存しました: {extended_filename}")
        print(f"拡張分析データ件数: {len(extended_df)}")
else:
    print("拡張処理はスキップされました。")
    print("実行する場合は RUN_EXTENDED = True に設定してください。")